# a)  Instalar dependencias del proyecto

In [1]:
# Instalar librerías de Python y Node.js para el túnel
!pip install -q streamlit pypdf2 langchain langchain-openai langchain-community langchain-huggingface faiss-cpu sentence-transformers langchain-core
!pip install -q langchain-google-genai
!pip install pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━

# b) Codigo del script  app.py  que se ejecutara con la logica del proyecto



In [5]:
%%writefile app.py
import streamlit as st
import PyPDF2
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
#from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
#from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI


# Configuración de la Ventana y Estado Global
# *******************************************************
st.set_page_config(page_title="Portal Multiagente de Reclutamiento", layout="wide")

if "vacantes" not in st.session_state:
    st.session_state.vacantes = []
if "candidato_stage" not in st.session_state:
    st.session_state.candidato_stage = "SETUP"
if "chat_historia" not in st.session_state:
    st.session_state.chat_historia = []
if "cv_text" not in st.session_state:
    st.session_state.cv_text = ""
if "vacante_seleccionada" not in st.session_state:
    st.session_state.vacante_seleccionada = None
if "contador_preguntas" not in st.session_state:
    st.session_state.contador_preguntas = 0

MAX_PREGUNTAS = 3

st.sidebar.title("Menu")

# Habilitar para usar GPT  ***********
#os.environ["OPENAI_API_KEY"] = st.sidebar.text_input("OpenAI API Key", type="password")
#api_key = os.environ.get("OPENAI_API_KEY")

# Cambio a uso de Gemini  ************
API_KEY_GEMINI = os.environ.get("GEMINI_API_KEY")

st.sidebar.markdown("---")
st.sidebar.subheader("Navegación de Roles")
menu_opcion = st.sidebar.radio("Selecciona tu interfaz:", ["Portal Reclutador", "Portal Candidato"])


# Embeddings
# *******************************************************
@st.cache_resource
def load_embedding_model():
    return HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

embeddings = load_embedding_model()


# Motores de IA
# *******************************************************
def extract_text_from_pdf(pdf_file):
    reader = PyPDF2.PdfReader(pdf_file)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

def buscar_vacante_idonea(cv_text, vacantes):
    textos_vacantes = [v["desc"] for v in vacantes]
    metadatos = [{"titulo": v["titulo"], "desc": v["desc"]} for v in vacantes]
    vectorstore = FAISS.from_texts(textos_vacantes, embeddings, metadatas=metadatos)
    resultado_busqueda = vectorstore.similarity_search_with_score(cv_text, k=1)
    doc_match, _ = resultado_busqueda[0]
    return doc_match.metadata

def obtener_agente_entrevistador(titulo_vacante, vacante_desc, cv):
    # Habilitar para usar GPT  ***********
    #llm = ChatOpenAI(temperature=0.5, model_name="gpt-4o-mini")
    llm = ChatGoogleGenerativeAI(temperature=0.5, model="models/gemini-2.5-flash", google_api_key=API_KEY_GEMINI)
    #modelo_flash = genai.GenerativeModel("models/gemini-3.1-flash-lite")

    prompt = ChatPromptTemplate.from_messages([
        ("system", f"""Eres el 'Agente Entrevistador'. Un candidato fue vinculado a la vacante: '{titulo_vacante}'.
        DETALLES DE LA VACANTE: {vacante_desc}
        CV DEL CANDIDATO: {cv}
        INSTRUCCIONES:
        1. Haz una sola pregunta a la vez.
        2. Reacciona a la respuesta del candidato para profundizar en detalles técnicos si es necesario.
        3. Mantén un tono profesional y fluido."""),
        MessagesPlaceholder(variable_name="history"),
    ])
    return prompt | llm

def ejecutar_agente_evaluador(transcripcion, cv, titulo_vacante, vacante_desc):
    # Habilitar para usar GPT  ***********
    #llm = ChatOpenAI(temperature=0.1, model_name="gpt-4o-mini")
    llm = ChatGoogleGenerativeAI(temperature=0.1, model="models/gemini-2.5-flash", google_api_key=API_KEY_GEMINI)

    prompt = """
    Eres el 'Agente Evaluador'. Genera un reporte analítico de la entrevista para el puesto: '{titulo_vacante}'.
    REQUISITOS DEL PUESTO: {vacante_desc}
    CV DEL CANDIDATO: {cv}
    TRANSCRIPCIÓN COMPLETA: {transcripcion}

    Estructura tu respuesta en Markdown incluyendo:
    - **Porcentaje de Compatibilidad Final** (0-100%).
    - **Puntos Fuertes Demostrados**.
    - **Areas de Riesgo / Brechas**.
    - **Dictamen Final**.
    """
    mensaje = prompt.format(titulo_vacante=titulo_vacante, vacante_desc=vacante_desc, cv=cv, transcripcion=transcripcion)
    return llm.invoke(mensaje).content


# UI
# *******************************************************
if menu_opcion == "Portal Reclutador":
    st.title("Panel de Reclutamiento: Carga de Vacantes")
    with st.form("nueva_vacante_form", clear_on_submit=True):
        titulo_puesto = st.text_input("Titulo de la Vacante:")
        descripcion_puesto = st.text_area("Descripción detallada del puesto (Requisitos, Skills):", height=200)
        if st.form_submit_button("Publicar Vacante"):
            if titulo_puesto and descripcion_puesto:
                st.session_state.vacantes.append({"titulo": titulo_puesto, "desc": descripcion_puesto})
                st.success(f"Vacante '{titulo_puesto}' guardada.")
            else:
                st.warning("Completa ambos campos.")

    st.markdown("---")
    if st.session_state.vacantes:
        for idx, vac in enumerate(st.session_state.vacantes):
            with st.expander(f"{idx + 1}. {vac['titulo']}"):
                st.write(vac["desc"])

elif menu_opcion == "Portal Candidato":
    st.title("Portal del Postulante: Evaluación Automatizada")
    if not st.session_state.vacantes:
        st.warning("El Reclutador no ha cargado ninguna vacante.")
    else:
        if st.session_state.candidato_stage == "SETUP":
            uploaded_cv = st.file_uploader("Cargar Currículum (PDF):", type="pdf")
            if st.button("Enviar Postulación e Iniciar", type="primary"):
                if not os.environ["GEMINI_API_KEY"]:
                    st.error("Se requiere la Gemini API Key")
                elif uploaded_cv:
                    with st.spinner("Analizando perfil..."):
                        st.session_state.cv_text = extract_text_from_pdf(uploaded_cv)
                        vacante_asignada = buscar_vacante_idonea(st.session_state.cv_text, st.session_state.vacantes)
                        st.session_state.vacante_seleccionada = vacante_asignada

                        entrevistador = obtener_agente_entrevistador(vacante_asignada["titulo"], vacante_asignada["desc"], st.session_state.cv_text)
                        primer_mensaje = entrevistador.invoke({"history": [HumanMessage(content="Hola, vengo a postularme.")]})

                        st.session_state.chat_historia.append(AIMessage(content=primer_mensaje.content))
                        st.session_state.contador_preguntas = 1
                        st.session_state.candidato_stage = "ENTREVISTA"
                        st.rerun()

        elif st.session_state.candidato_stage == "ENTREVISTA":
            st.success(f"**Vacante Asignada:** {st.session_state.vacante_seleccionada['titulo']}")
            for msg in st.session_state.chat_historia:
                if isinstance(msg, AIMessage):
                    with st.chat_message("assistant"): st.write(msg.content)
                elif isinstance(msg, HumanMessage):
                    with st.chat_message("user"): st.write(msg.content)

            if respuesta_usuario := st.chat_input("Escribe tu respuesta técnica aquí..."):
                st.session_state.chat_historia.append(HumanMessage(content=respuesta_usuario))
                if st.session_state.contador_preguntas < MAX_PREGUNTAS:
                    with st.spinner("Analizando respuesta..."):
                        entrevistador = obtener_agente_entrevistador(st.session_state.vacante_seleccionada["titulo"], st.session_state.vacante_seleccionada["desc"], st.session_state.cv_text)
                        sgte_pregunta = entrevistador.invoke({"history": st.session_state.chat_historia})
                        st.session_state.chat_historia.append(AIMessage(content=sgte_pregunta.content))
                        st.session_state.contador_preguntas += 1
                    st.rerun()
                else:
                    st.session_state.candidato_stage = "EVALUACION"
                    st.rerun()

        elif st.session_state.candidato_stage == "EVALUACION":
            with st.spinner("Generando reporte..."):
                transcripcion = "\n\n".join([f"{'Agente' if isinstance(msg, AIMessage) else 'Candidato'}: {msg.content}" for msg in st.session_state.chat_historia])
                reporte = ejecutar_agente_evaluador(transcripcion, st.session_state.cv_text, st.session_state.vacante_seleccionada["titulo"], st.session_state.vacante_seleccionada["desc"])
                st.markdown(reporte)

            if st.button("Finalizar y Reiniciar Sesión"):
                st.session_state.candidato_stage = "SETUP"
                st.session_state.chat_historia = []
                st.session_state.cv_text = ""
                st.session_state.vacante_seleccionada = None
                st.session_state.contador_preguntas = 0
                st.rerun()

Overwriting app.py


# c) Declarar y abrir tunel para conectar mediante Streamlit

In [3]:
from pyngrok import ngrok

# Opcional: Si tienes un token de authtoken de ngrok, colócalo aquí
from google.colab import userdata

ngrok.set_auth_token(userdata.get('ngrok'))

public_url = ngrok.connect(8501)
print(f"Tu aplicación está en: {public_url.public_url}")

Tu aplicación está en: https://pug-impeach-zipfile.ngrok-free.dev


# d) Ejecutar con Streamlit

In [6]:
from google.colab import userdata
import os

# 1. Obtenemos el secreto del Colab
#mi_api_key = userdata.get('OpenAI')
mi_api_key = userdata.get("GOOGLE_API_KEY")

# 2. Exportamos la variable de entorno para que el proceso de Streamlit la herede
#os.environ["OPENAI_API_KEY"] = mi_api_key
os.environ["GEMINI_API_KEY"] = mi_api_key

!streamlit run app.py &>/content/logs.txt &

# Apoyo - Ver modelos disponibles de Gemini de acuerdo a la licencia y creditos

In [17]:
import google.generativeai as genai
import os

# Configura tu llave
genai.configure(api_key=userdata.get("GOOGLE_API_KEY"))

# Esto te imprimirá exactamente cómo se llaman los modelos que Google te permite usar
print("Modelos disponibles:")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(f"- {m.name}")

Modelos disponibles:
- models/gemini-2.5-flash
- models/gemini-2.5-pro
- models/gemini-2.0-flash
- models/gemini-2.0-flash-001
- models/gemini-2.0-flash-lite-001
- models/gemini-2.0-flash-lite
- models/gemini-2.5-flash-preview-tts
- models/gemini-2.5-pro-preview-tts
- models/gemma-4-26b-a4b-it
- models/gemma-4-31b-it
- models/gemini-flash-latest
- models/gemini-flash-lite-latest
- models/gemini-pro-latest
- models/gemini-2.5-flash-lite
- models/gemini-2.5-flash-image
- models/gemini-3-pro-preview
- models/gemini-3-flash-preview
- models/gemini-3.1-pro-preview
- models/gemini-3.1-pro-preview-customtools
- models/gemini-3.1-flash-lite-preview
- models/gemini-3.1-flash-lite
- models/gemini-3-pro-image-preview
- models/gemini-3-pro-image
- models/nano-banana-pro-preview
- models/gemini-3.1-flash-image-preview
- models/gemini-3.1-flash-image
- models/gemini-3.1-flash-lite-image
- models/gemini-3.5-flash
- models/gemini-omni-flash-preview
- models/lyria-3-clip-preview
- models/lyria-3-pro-pr